In [ ]:
# Here i did not rely on ChatGPT, i built it myself to learn and understand the concepts, Use AI as a tool.. not a replacement for learning.


In [1]:
!pip install pypdf sentence-transformers 

In [2]:
!pip install faiss-cpu


In [3]:
import os
import numpy as np
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
from tqdm import tqdm
import pickle


/home/neelamlab/anaconda3/envs/neelam_lab/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# The first step is to extract the information, Here iam extracting text from pdf
# 
def ext_text(path):
    data = PdfReader(path)
    df=""
    for x in data.pages:
        x_text=x.extract_text()
        if x_text:
            df += x_text +"\n"

    return df        



In [ ]:
path="/data/LM/rag_project/notes.pdf"
data = ext_text(path)
data

'Introduction to Machine Learning Class Notes\nHuy Nguyen\nPhD Student, Human-Computer Interaction Institute\nCarnegie Mellon University\nContents\nPreface 3\n1 MLE and MAP 4\n1.1 MLE . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n1.2 Bayesian learning and MAP . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5\n2 Nonparametric models: KNN and kernel regression 7\n2.1 Bayes decision rule . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 7\n2.2 Classiﬁcation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 8\n2.3 K-nearest neighbors . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 8\n2.4 Local Kernel Regression . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 9\n3 Linear Regression 11\n3.1 Basic linear regression . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 11\n3.2 Multivariate and general linear regression . . . . . . . . . . . . . . . . . . . . 

In [7]:
# Here we will chunk the text with over lap
def chunk_text(text, chunk_size=500,overlap=100):
    chunks=[]
    start=0 
    while start < len(text):
        end =  start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size-overlap
    return chunks

In [8]:
chunks=chunk_text(data)
print(len(chunks),chunks)

231 ['Introduction to Machine Learning Class Notes\nHuy Nguyen\nPhD Student, Human-Computer Interaction Institute\nCarnegie Mellon University\nContents\nPreface 3\n1 MLE and MAP 4\n1.1 MLE . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n1.2 Bayesian learning and MAP . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 5\n2 Nonparametric models: KNN and kernel regression 7\n2.1 Bayes decision rule . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 7\n2.', 'n 7\n2.1 Bayes decision rule . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 7\n2.2 Classiﬁcation . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 8\n2.3 K-nearest neighbors . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 8\n2.4 Local Kernel Regression . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 9\n3 Linear Regression 11\n3.1 Basic linear regression . . . . . . . . . . . . . . . . . . . . . . 

In [17]:
import os
import torch
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [18]:
## Load Embedding Model
## emb =  SentenceTransformer("BAAI/bge-large-en-v1.5",device="cuda:6")
emb = SentenceTransformer("all-MiniLM-L6-v2", device="cuda:2")   

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 554.11it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
# Here we will create embeddings using the above model
embeddings = emb.encode(chunks,batch_size=10,convert_to_numpy=True,
show_progress_bar=True)
# here we convert each chunk into vector

Batches:  21%|██        | 5/24 [00:00<00:00, 45.90it/s]

Batches: 100%|██████████| 24/24 [00:00<00:00, 58.77it/s]


In [21]:
embeddings

array([[ 0.00338129, -0.09022815,  0.08497572, ..., -0.02394508,
        -0.08998082,  0.04077885],
       [-0.02748079, -0.02066522,  0.01795669, ..., -0.01886765,
        -0.01819193,  0.01213914],
       [ 0.0769441 , -0.06430613, -0.05270697, ...,  0.01947873,
        -0.0101842 , -0.04567917],
       ...,
       [-0.08240927,  0.00592057,  0.03266907, ...,  0.00999525,
         0.06170399, -0.02119865],
       [ 0.00646808,  0.04383314, -0.00831901, ..., -0.03141473,
         0.04569596, -0.02072829],
       [ 0.05067067, -0.03856865,  0.06469955, ..., -0.05472222,
        -0.05902956, -0.03183942]], dtype=float32)

In [22]:
# Here each chunk is 384 number, each chunk is one point in High dimensional space
# Faiss stores those points so we can ask later
embeddings[0].shape

(384,)

In [ ]:
#--------- Build FAISS Index --------####
# here we store the emeddings (vectors) in high dimensional space 

In [23]:
dimension = embeddings.shape[1]
# we create faiss index (nocompression with L2 distance)
index=faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [24]:
embeddings.shape

(231, 384)

In [ ]:
faiss.write_index(index, "/data/LM/rag_project/faiss_index.bin")
with open("/data/LM/rag_project/chunks.pkl", "wb") as f:
    pickle.dump(chunks, f)

Question Answering (RAG + LLM).

In [ ]:
index = faiss.read_index("/data/LM/rag_project/faiss_index.bin")

with open("/data/LM/rag_project/chunks.pkl", "rb") as f:
    chunks = pickle.load(f)


In [29]:
## Load Small LLM
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)

model = model.to("cuda:2")   

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 201/201 [00:08<00:00, 22.70it/s, Materializing param=model.norm.weight]                              


In [30]:
print(model)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

In [31]:
### Retrieval Function
def retrieve(q,k=3):
    q_emb=emb.encode([q],convert_to_numpy=True)
    D,I=index.search(q_emb,k)
    return [chunks[i] for i in I[0]]

##Question → embedding

##FAISS finds nearest chunks

##Returns top-k most relevant text pieces

##This is pure RAG retrieval.

In [34]:
## Generate Answer Function

def generate_answer(q):
    retrieve_chunk=retrieve(q)
    context="\n\n".join(retrieve_chunk)
    prompt=f""" You are a helpful Machine learning assistant.
    Answer ONLY using the provided CONTEXT.
    If answer not found, say "NOt found in Notes"

    Context:{context}
    Question:{q}
    Answer:
    """

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:2")

    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.3,
        do_sample=True
    )

    # Only decode newly generated tokens
    generated_tokens = output[0][inputs["input_ids"].shape[1]:]

    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    return answer.strip()

    

In [36]:
while True:
    question = input("\nAsk: ")

    if question.lower() in ["exit", "quit"]:
        break

    answer = generate_answer(question)
    print("question:",question)
    print("\nAnswer:\n")
    print(answer)


question: what is support vector machine?

Answer:

1. A supervised learning algorithm that uses a kernel trick to transform inputs into a high
       dimensional space where the decision boundary is a hyperplane.
    2. It is a non-linear classifier that can handle non-separable data.
    3. It is a versatile algorithm that can be used for classification, regression, clustering,
       and many other tasks.
    4. It is a popular technique in machine learning, especially in the field of computer vision.
    5. It is a popular technique in the field of natural language processing.
    6. It is used in many other fields such as bioinformatics, finance, and robotics.
    7. It is a popular technique in the field of computer vision.
    8. It is a popular technique in the field of natural language processing.
    9. It is a popular technique in the field of bioinformatics.
    10. It is a popular technique in the field of finance.
    11. It is a popular technique in the field of robotics

In [ ]:
# This is we have learn from previous post, now improve its performance

we want to test the same question:

“What is support vector machine?”

and understand how these parameters change behavior:

> do_sample

> temperature

> repetition_penalty

> max_new_tokens

First: What Each Parameter Controls we need to understand it.

Before running experiments, understand conceptually:

**Parameter**       	        **Controls**	              **Affects**

do_sample	                Randomness mode                Deterministic vs stochastic
temperature	              Randomness intensity	             Creativity level
repetition_penalty	        Loop prevention	                 Repeated words/lists
max_new_tokens	            Output length	                 Rambling vs concise

In [ ]:
#We will test 4 configurations for the SAME question.

In [39]:
def test_generation(q, temperature, do_sample, repetition_penalty, max_new_tokens):

    retrieved_chunk = retrieve(q)
    context = "\n\n".join(retrieved_chunk)

    prompt = f"""
You are a helpful Machine learning assistant.
    Answer ONLY using the provided CONTEXT.
    If answer not found, say "NOt found in Notes."

Context:
{context}

Question:
{q}

Answer:
"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda:2")

    output = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=do_sample,
        repetition_penalty=repetition_penalty
    )

    generated_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(generated_tokens, skip_special_tokens=True)

    print("\n============================")
    print(f"temperature={temperature}")
    print(f"do_sample={do_sample}")
    print(f"repetition_penalty={repetition_penalty}")
    print(f"max_new_tokens={max_new_tokens}")
    print("============================\n")
    print(answer)

In [ ]:
#Deterministic (Best for RAG)
test_generation(
    "What is support vector machine?",
    temperature=0.0,
    do_sample=False,
    repetition_penalty=1.2,
    max_new_tokens=200
)
#Expected: Stable,Less creative,More grounded,Shorter answer


temperature=0.0
do_sample=False
repetition_penalty=1.2
max_new_tokens=200

Support Vector Machines (SVMs) are a family of supervised classification algorithms based on the principle of separation between classes. It uses a kernel trick to map the input space into an inner product space where the distance between two points is measured by their squared norm. This leads to a high degree of separation between classes.


In [44]:
#Mild Creativity
test_generation(
    "What is support vector machine?",
    temperature=0.3,
    do_sample=True,
    repetition_penalty=1.2,
    max_new_tokens=200
)
#Expected: Slight variation, Possibly more explanation, Slight hallucination risk


temperature=0.3
do_sample=True
repetition_penalty=1.2
max_new_tokens=200

Support Vector Machines (SVMs) are a family of supervised classification algorithms based on the idea of constructing hyperplanes which separate the training data into two classes. This allows us to achieve high accuracy while avoiding overfitting.


In [ ]:
#High Creativity
#Expected:More general knowledge,More hallucination,
#Longer output

Possible repetition
test_generation(
    "What is support vector machine?",
    temperature=0.9,
    do_sample=True,
    repetition_penalty=1.0,
    max_new_tokens=300
)


temperature=0.9
do_sample=True
repetition_penalty=1.0
max_new_tokens=300

Support vector machine is a machine learning technique that can be used for classification and
regression tasks. It is a branch of machine learning that involves the optimization of
decision boundaries between classes. It differs from other machine learning algorithms in that it
uses linear separators, not quadratic ones, to describe the decision boundary and not the range of
the decision function. The approach relies on the idea of a "kernel" that provides a similarity
between samples for calculating the decision boundary. With this algorithm, one performs a
two-step process:

1. First, one computes a hyperplane that separates the data into two classes.
2. Then, one finds the maximum margin hyperplane.

8.2 Ensemble Methods
Ensemble methods are a class of algorithms that combine the predictions of multiple learners to
provide more accurate predictions. 
8.2.1 Combining Learners by bagging

bagging is a simple e

In [46]:
#Strong Anti-Repetition
test_generation(
    "What is support vector machine?",
    temperature=0.0,
    do_sample=False,
    repetition_penalty=1.7,
    max_new_tokens=150
)


temperature=0.0
do_sample=False
repetition_penalty=1.7
max_new_tokens=150

Support Vector Machines(SVM), an algorithm used by computer vision researcher Vapnik et al.. It's based on kernel methods which use similarity between points instead of distance measure like Euclidean or Manhattan distances. This makes it more flexible compared other traditional classification algorithms such as Naïve bayesian model where you need specific feature space dimensions before training your models because they assume all possible combinations exist within each dimension so if one variable changes then there will be some combination change across different variables resulting into noisy output from any given point regardless how many classes were defined initially! In contrast SupportVectorMachine uses multiple hyperplanar regions called supports around every sampled instance; when two instances cross their respective boundaries either inside another region/support boundary


## What You Should Observe During Experiments

### 🔹 Effect of `do_sample`

- **False** → Same output every run (deterministic)
- **True** → Different output each run (stochastic / random sampling)

---

### 🔹 Effect of `temperature`

**Low (0.0)**  
- Safe  
- Conservative  
- Deterministic  
- Less hallucination  

**High (0.9)**  
- Expands explanation  
- More creative  
- Higher hallucination risk  
- May introduce outside knowledge  

---

### 🔹 Effect of `repetition_penalty`

**Low (1.0)**  
- May repeat phrases  
- May generate looping lists  

**High (1.7)**  
- More concise  
- Reduces repetition  
- Can sometimes create slightly unnatural phrasing  

---

### 🔹 Effect of `max_new_tokens`

**Small (100–150)**  
- Compact answers  
- Less rambling  
- More focused  

**Large (300+)**  
- Longer explanations  
- May ramble  
- Higher chance of list-style repetition  

In [ ]:
What This Experiment Teaches:

we just discovered:

🔹 1. RAG stability depends heavily on decoding strategy
🔹 2. High temperature causes:

Topic drift

Hallucination

Context bleeding

🔹 3. High max_new_tokens causes:

Long runaway outputs

Model continuing beyond answer

🔹 4. Too high repetition penalty:

Makes language unnatural

Can distort response